# DK-SOFNN — Algorithm Box Equations Only

**Paper:** Han, H., Liu, H., & Qiao, J. (2024). Data-Knowledge-Driven Self-Organizing Fuzzy Neural Network.  
*IEEE Transactions on Neural Networks and Learning Systems*, 35(2), 2081–2095.

This notebook implements **only** the numbered equations that appear in the Algorithm II box (Table II of the paper), exactly as listed:

| Step | Description | Equations |
|------|-------------|-----|
| 5  | Optimize parameters of source FNN | Eqs. (7–9) |
| 6  | Grow fuzzy rule in growing phase | Eqs. (17)–(18) |
| 7  | Prune fuzzy rule in pruning phase | Eq. (19) |
| 8  | Replace fuzzy rule in compensating phase | Eqs. (20)–(21) |
| 9  | Keep fuzzy rule in constant phase | Eq. (22) |
| 10 | Compute criterion function | Eq. (23) |
| 12 | Generate parameter groups | Eqs. (24)–(28) |
| 13 | Select optimal parameters | Eqs. (29)–(30) |


In [ ]:
import numpy as np

---
## Step 5 — Eqs. (7)–(9): Source FNN Output, Back-Propagation Update, Error Function

**Eq. (7)** — Source FNN output (RBF weighted sum):
$$y^S(n) = \frac{\sum_{k=1}^{K} w_k^S(n) \prod_{p=1}^{P} \exp\!\left(-\dfrac{(x_p^S(n) - c_{pk}^S(n))^2}{2(\sigma_{pk}^S(n))^2}\right)}{\sum_{k=1}^{K} \prod_{p=1}^{P} \exp\!\left(-\dfrac{(x_p^S(n) - c_{pk}^S(n))^2}{2(\sigma_{pk}^S(n))^2}\right)}$$

**Eq. (8)** — Back-propagation parameter update for source FNN:
$$\Theta^S(n+1) = \Theta^S(n) - \lambda_S \frac{\partial E^S(n)}{\partial \Theta^S(n)}$$

**Eq. (9)** — Squared-error objective of source FNN:
$$E^S(n) = \frac{1}{2}\,e^S(n)^2, \qquad e^S(n) = y^S(n) - y^S_d(n)$$

In [ ]:
# ============================================================
# Eqs. (7)–(9): Source FNN — forward pass + BP parameter update
# ============================================================

def source_fnn_forward(x, C, S, W):
    """
    Eq. (7): Compute source FNN output y_S and per-rule activations.

    Parameters
    ----------
    x : (P,)   input vector
    C : (K, P) rule centres
    S : (K, P) rule widths  (sigma)
    W : (K,)   rule weights

    Returns
    -------
    y_S : float   FNN output
    u   : (K,)    unnormalised Gaussian activations
    v   : (K,)    normalised firing strengths
    """
    # Gaussian activations (product over P dimensions)
    diff = x[None, :] - C                              # (K, P)
    u = np.exp(-0.5 * ((diff / (S + 1e-8)) ** 2).sum(axis=1))  # (K,)
    v = u / (u.sum() + 1e-10)                          # normalised (K,)
    y_S = float(W @ v)                                 # Eq. (7)
    return y_S, u, v


def source_fnn_error(y_S, y_S_d):
    """
    Eq. (9): Source FNN squared-error objective.

    Returns E_S (scalar) and e_S (prediction error).
    """
    e_S = y_S - y_S_d          # output error
    E_S = 0.5 * e_S ** 2       # Eq. (9)
    return E_S, e_S


def source_fnn_update(x, y_S, e_S, v, C, S, W, lr):
    """
    Eq. (8): Back-propagation update for source FNN parameters.
    Theta_S = [C, S, W].  Updates are applied in-place.

    Returns updated C, S, W.
    """
    K = len(W)

    # --- Gradient w.r.t. weights (dE/dW = e_S * v) ---
    dW = e_S * v                                       # (K,)

    # --- Gradients w.r.t. centres and widths ---
    dC = np.zeros_like(C)
    dS = np.zeros_like(S)
    for k in range(K):
        s2 = S[k] ** 2 + 1e-8
        s3 = S[k] ** 3 + 1e-8
        dC[k] = e_S * v[k] * (W[k] - y_S) * (x - C[k]) / s2
        dS[k] = e_S * v[k] * (W[k] - y_S) * (x - C[k]) ** 2 / s3

    # --- Eq. (8): Theta_S(n+1) = Theta_S(n) - lr * dE/dTheta_S ---
    W = W - lr * dW
    C = C - lr * dC
    S = np.maximum(S - lr * dS, 0.01)
    return C, S, W


# ---- Quick smoke-test ----
K, P = 5, 4
C_s = np.random.rand(K, P)
S_s = np.full((K, P), 0.3)
W_s = np.random.randn(K) * 0.1
x_s = np.random.rand(P)
y_d  = 0.5

y_S, u_s, v_s = source_fnn_forward(x_s, C_s, S_s, W_s)     # Eq. 7
E_S, e_S      = source_fnn_error(y_S, y_d)                   # Eq. 9
C_s, S_s, W_s = source_fnn_update(x_s, y_S, e_S, v_s,       # Eq. 8
                                   C_s, S_s, W_s, lr=0.05)
print(f"Eq.(7)  y_S = {y_S:.4f}")
print(f"Eq.(9)  E_S = {E_S:.6f},  e_S = {e_S:.4f}")
print(f"Eq.(8)  Source FNN updated — new W[0] = {W_s[0]:.6f}")

---
## Step 6 — Eqs. (17)–(18): Growing Phase — Condition + New Rule Initialisation

**Eq. (17)** — Growing condition (all three must hold simultaneously):
$$\bar{R}_T(t) \leq \bar{R}_S(n), \quad \bar{M}_T(t) \geq \bar{M}_S(n), \quad \bar{C}_T(t) \leq \bar{C}_S(n)$$

**Eq. (18)** — New rule initialisation (from rule $l$ with largest contribution):
$$c^{\text{new}}_{T,q}(t) = x_{T,q}(t), \quad
\sigma^{\text{new}}_{T,q}(t) = \frac{|x_{T,q}(t) - c^l_{T,q}(t)|}{2}, \quad
w^{\text{new}}_T(t) = w^l_T(t)$$

In [ ]:
# ============================================================
# Eqs. (17)–(18): Growing Phase
# ============================================================

def growing_condition(Rbar_T, Mbar_T, Cbar_T, Rbar_S, Mbar_S, Cbar_S):
    """
    Eq. (17): Return True if the target FNN satisfies the growing condition.

    Rbar_T/S : mean similarity of target / source fuzzy rules
    Mbar_T/S : mean sensitivity
    Cbar_T/S : mean contribution
    """
    return (Rbar_T <= Rbar_S) and (Mbar_T >= Mbar_S) and (Cbar_T <= Cbar_S)


def grow_new_rule(x_T, C_T, S_T, W_T, l):
    """
    Eq. (18): Add a new rule to the target FNN, seeded from rule l.

    Parameters
    ----------
    x_T : (Q,)   current target input
    C_T : (B, Q) target rule centres
    S_T : (B, Q) target rule widths
    W_T : (B,)   target rule weights
    l   : int    index of rule with largest contribution

    Returns updated C_T, S_T, W_T with one extra rule appended.
    """
    c_new = x_T.copy()                                # Eq. (18) — centre
    s_new = np.abs(x_T - C_T[l]) / 2.0 + 1e-4       # Eq. (18) — width
    w_new = W_T[l]                                    # Eq. (18) — weight

    C_T = np.vstack([C_T, c_new])
    S_T = np.vstack([S_T, s_new])
    W_T = np.append(W_T, w_new)
    return C_T, S_T, W_T


# ---- Quick smoke-test ----
B, Q = 4, 4
C_T = np.random.rand(B, Q)
S_T = np.full((B, Q), 0.3)
W_T = np.random.randn(B) * 0.1
x_T = np.random.rand(Q)

Rbar_S, Mbar_S, Cbar_S = 1.5, 0.3, 0.4
Rbar_T, Mbar_T, Cbar_T = 1.2, 0.5, 0.2  # satisfies Eq. (17)

if growing_condition(Rbar_T, Mbar_T, Cbar_T, Rbar_S, Mbar_S, Cbar_S):  # Eq. 17
    l_best = 0  # rule with largest contribution (placeholder)
    C_T, S_T, W_T = grow_new_rule(x_T, C_T, S_T, W_T, l_best)           # Eq. 18
    print(f"Eq.(17) Growing condition satisfied.")
    print(f"Eq.(18) New rule added → total rules = {len(W_T)}")
else:
    print("Growing condition NOT satisfied.")

---
## Step 7 — Eq. (19): Pruning Phase — Condition

**Eq. (19)** — Pruning condition (either sub-condition triggers pruning):
$$\frac{\bar{R}_T(t) \geq \bar{R}_S(n) \;\text{ AND }\; \bar{M}_T(t) \leq \bar{M}_S(n)}{\text{OR}}$$
$$\frac{\bar{C}_T(t) \geq \bar{C}_S(n) \;\text{ AND }\; \bar{C}_T(t) \geq \bar{C}_S(n)}{}$$

More precisely (as in the paper):
$$\left[\bar{R}_T(t)\geq\bar{R}_S(n),\;\bar{M}_T(t)\leq\bar{M}_S(n)\right]\;\text{OR}\;\left[\bar{C}_T(t)\geq\bar{C}_S(n),\;\bar{C}_T(t)\geq\bar{C}_S(n)\right]$$

In [ ]:
# ============================================================
# Eq. (19): Pruning Phase Condition
# ============================================================

def pruning_condition(Rbar_T, Mbar_T, Cbar_T, Rbar_S, Mbar_S, Cbar_S):
    """
    Eq. (19): Return True if the target FNN satisfies the pruning condition.

    Two sub-conditions joined by OR:
      (a) higher similarity AND lower sensitivity than source
      (b) higher similarity AND higher contribution than source
    """
    cond_a = (Rbar_T >= Rbar_S) and (Mbar_T <= Mbar_S)   # Eq. (19) branch 1
    cond_b = (Cbar_T >= Cbar_S) and (Cbar_T >= Cbar_S)   # Eq. (19) branch 2
    return cond_a or cond_b


def prune_rule(C_T, S_T, W_T, l):
    """
    Remove rule l from the target FNN (parameters reset to zero then deleted).
    """
    C_T = np.delete(C_T, l, axis=0)
    S_T = np.delete(S_T, l, axis=0)
    W_T = np.delete(W_T, l)
    return C_T, S_T, W_T


# ---- Quick smoke-test ----
Rbar_T2, Mbar_T2, Cbar_T2 = 2.0, 0.1, 0.6  # satisfies Eq. (19) branch (a)

if pruning_condition(Rbar_T2, Mbar_T2, Cbar_T2, Rbar_S, Mbar_S, Cbar_S):  # Eq. 19
    l_worst = 1  # rule with worst performance (placeholder)
    C_T, S_T, W_T = prune_rule(C_T, S_T, W_T, l_worst)
    print(f"Eq.(19) Pruning condition satisfied.")
    print(f"        Rule {l_worst} pruned → remaining rules = {len(W_T)}")
else:
    print("Pruning condition NOT satisfied.")

---
## Step 8 — Eqs. (20)–(21): Compensating Phase — Condition + Rule Replacement

**Eq. (20)** — Compensating condition (either sub-condition triggers compensation):
$$\left[\bar{R}_T(t)\geq\bar{R}_S(n),\;\bar{M}_T(t)\leq\bar{M}_S(n)\right]\;\text{OR}\;\left[\bar{C}_T(t)\leq\bar{C}_S(n),\;\bar{C}_T(t)\leq\bar{C}_S(n)\right]$$

**Eq. (21)** — Reset target rule $l$ with source rule $z$ (best performance):
$$c^l_T(t) = c^z_S(n), \quad \sigma^l_T(t) = \sigma^z_S(n), \quad w^l_T(t) = w^z_S(n)$$

In [ ]:
# ============================================================
# Eqs. (20)–(21): Compensating Phase
# ============================================================

def compensating_condition(Rbar_T, Mbar_T, Cbar_T, Rbar_S, Mbar_S, Cbar_S):
    """
    Eq. (20): Return True if the target FNN satisfies the compensating condition.

    Two sub-conditions joined by OR:
      (a) higher similarity AND lower sensitivity than source
      (b) lower contribution than source (in either metric)
    """
    cond_a = (Rbar_T >= Rbar_S) and (Mbar_T <= Mbar_S)   # Eq. (20) branch 1
    cond_b = (Cbar_T <= Cbar_S) and (Cbar_T <= Cbar_S)   # Eq. (20) branch 2
    return cond_a or cond_b


def replace_rule(C_T, S_T, W_T, l, c_src_z, s_src_z, w_src_z):
    """
    Eq. (21): Replace target rule l with source rule z.

    Parameters
    ----------
    l          : index of worst-performing target rule
    c_src_z    : (Q,) centre of best source rule z
    s_src_z    : (Q,) width  of best source rule z
    w_src_z    : float weight of best source rule z

    Returns updated C_T, S_T, W_T.
    """
    C_T[l] = c_src_z.copy()   # Eq. (21)
    S_T[l] = s_src_z.copy()   # Eq. (21)
    W_T[l] = w_src_z          # Eq. (21)
    return C_T, S_T, W_T


# ---- Quick smoke-test ----
Rbar_T3, Mbar_T3, Cbar_T3 = 2.0, 0.1, 0.2  # satisfies Eq. (20)

if compensating_condition(Rbar_T3, Mbar_T3, Cbar_T3, Rbar_S, Mbar_S, Cbar_S):  # Eq. 20
    l_worst = 0              # worst target rule (placeholder)
    z_best  = 2              # best  source rule (placeholder)
    C_T, S_T, W_T = replace_rule(C_T, S_T, W_T, l_worst,
                                  C_s[z_best], S_s[z_best], W_s[z_best])  # Eq. 21
    print(f"Eq.(20) Compensating condition satisfied.")
    print(f"Eq.(21) Rule {l_worst} replaced with source rule {z_best}.")
else:
    print("Compensating condition NOT satisfied.")

---
## Step 9 — Eq. (22): Constant Phase — Condition

**Eq. (22)** — Constant condition (structure unchanged; all three must hold):
$$\bar{R}_T(t) \geq \bar{R}_S(n), \quad \bar{M}_T(t) \leq \bar{M}_S(n), \quad \bar{C}_T(t) \geq \bar{C}_S(n)$$

In [ ]:
# ============================================================
# Eq. (22): Constant Phase Condition
# ============================================================

def constant_condition(Rbar_T, Mbar_T, Cbar_T, Rbar_S, Mbar_S, Cbar_S):
    """
    Eq. (22): Return True if the target FNN structure should remain unchanged.

    Higher similarity, lower sensitivity, higher contribution than source
    → strong generalization AND high accuracy → no adjustment needed.
    """
    return (Rbar_T >= Rbar_S) and (Mbar_T <= Mbar_S) and (Cbar_T >= Cbar_S)


# ---- Quick smoke-test ----
Rbar_T4, Mbar_T4, Cbar_T4 = 2.0, 0.2, 0.6   # satisfies Eq. (22)

if constant_condition(Rbar_T4, Mbar_T4, Cbar_T4, Rbar_S, Mbar_S, Cbar_S):  # Eq. 22
    print("Eq.(22) Constant condition satisfied — structure unchanged.")
else:
    print("Constant condition NOT satisfied.")

---
## Step 10 — Eq. (23): Criterion Function $E^h_T(t)$

**Eq. (23)** — The $h$-th least-square criterion function of the target FNN:
$$E^h_T(t) = \alpha^h(t)\bigl(y_T(t) - y_{Td}(t)\bigr)^2 + \beta^h(t)\bigl(\Theta_T(t) - K_S(n)\bigr)^2$$

where $\alpha^h(t)$ is the data-driven weight, $\beta^h(t)$ is the knowledge-driven weight,  
$\Theta_T(t)$ is the parameter vector of the target FNN, and $K_S(n)$ is the source knowledge.

In [ ]:
# ============================================================
# Eq. (23): Criterion function E_T^h(t)
# ============================================================

def criterion_function(y_T, y_Td, Theta_T, K_S, alpha_h, beta_h):
    """
    Eq. (23): Combined data-driven + knowledge-driven criterion.

    Parameters
    ----------
    y_T     : float   target FNN output at time t
    y_Td    : float   desired (true) output at time t
    Theta_T : (D,)    flattened parameter vector of target FNN [C, S, W]
    K_S     : (D,)    flattened source knowledge (parameter vector of source FNN)
    alpha_h : float   data-driven weight for framework h
    beta_h  : float   knowledge-driven weight for framework h

    Returns
    -------
    E_h : float   criterion value
    """
    data_term      = alpha_h * (y_T - y_Td) ** 2               # Eq. (23) — data term
    n = min(len(Theta_T), len(K_S))
    knowledge_term = beta_h  * np.sum((Theta_T[:n] - K_S[:n]) ** 2)  # Eq. (23) — knowledge term
    E_h = data_term + knowledge_term                            # Eq. (23)
    return E_h


# ---- Quick smoke-test ----
Theta_T_vec = np.concatenate([C_T.ravel(), S_T.ravel(), W_T])
K_S_vec     = np.concatenate([C_s.ravel(), S_s.ravel(), W_s])
alpha_h, beta_h = 0.8, 0.2
y_T_val, y_Td_val = 0.45, 0.50

E_h = criterion_function(y_T_val, y_Td_val, Theta_T_vec, K_S_vec, alpha_h, beta_h)  # Eq. 23
print(f"Eq.(23) E_h = {E_h:.6f}  (alpha={alpha_h}, beta={beta_h})")

---
## Step 12 — Eqs. (24)–(28): Generate $H$ Parameter Groups + Value Function

**Eq. (24)** — Back-propagation update for framework $h$:
$$\Theta^h_T(t+1) = \Theta_T(t) - \lambda_T \frac{\partial E^h_T(t)}{\partial \Theta_T(t)}$$

**Eq. (25)** — Gradient vector (stacked over all rules):
$$\frac{\partial E^h_T(t)}{\partial \Theta_T(t)} = \left[\frac{\partial E^h_T(t)}{\partial k^1_T(t)}^T, \ldots, \frac{\partial E^h_T(t)}{\partial k^B_T(t)}^T\right]^T$$

**Eq. (26)** — Per-rule gradient:
$$\frac{\partial E^h_T(t)}{\partial k^b_T(t)} = \left[\frac{\partial E^h_T(t)}{\partial c^b_T(t)}^T,\; \frac{\partial E^h_T(t)}{\partial \sigma^b_T(t)}^T,\; \frac{\partial E^h_T(t)}{\partial w^b_T(t)}\right]^T$$

**Eq. (27)** — Explicit partial derivatives:
$$\frac{\partial E^h_T}{\partial c^b_T} = \alpha^h e_T \cdot c(t) + \beta^h(c^b_T - c^b_S)$$
$$\frac{\partial E^h_T}{\partial \sigma^b_T} = \alpha^h e_T \cdot \sigma(t) + \beta^h(\sigma^b_T - \sigma^b_S)$$
$$\frac{\partial E^h_T}{\partial w^b_T} = \alpha^h e_T \cdot v^b(t) + \beta^h(w^b_T - w^b_S)$$

**Eq. (28)** — Value function (expected future error for framework $h$):
$$W^h(t) = \mathbb{E}\bigl[e^h_T(t), \ldots, e^h_T(t+N) \mid \Theta^h_T(t+1)\bigr]$$

In [ ]:
# ============================================================
# Eqs. (24)–(28): Generate H parameter groups and compute value function
# ============================================================

def compute_gradients(x_T, y_T, y_Td, v_T, C_T, S_T, W_T,
                      C_S, S_S, W_S, alpha_h, beta_h):
    """
    Eqs. (25)–(27): Compute gradient of E_T^h w.r.t. all target FNN parameters.

    Returns
    -------
    dC : (B, Q)  gradient w.r.t. centres
    dS : (B, Q)  gradient w.r.t. widths
    dW : (B,)    gradient w.r.t. weights
    """
    B = len(W_T)
    e_T = y_T - y_Td                               # output error
    dC = np.zeros_like(C_T)
    dS = np.zeros_like(S_T)
    dW = np.zeros(B)

    for b in range(B):
        s2 = S_T[b] ** 2 + 1e-8
        s3 = S_T[b] ** 3 + 1e-8
        # Short-term variables for centres and widths (Eq. 27 notation)
        c_short = v_T[b] * (W_T[b] - y_T) * (x_T - C_T[b]) / s2   # c(t) in Eq.27
        s_short = v_T[b] * (W_T[b] - y_T) * (x_T - C_T[b])**2 / s3  # sigma(t) in Eq.27

        kb = min(b, len(C_S) - 1)  # align source rule index

        # Eq. (27) — partial derivatives
        dC[b] = alpha_h * e_T * c_short + beta_h * (C_T[b] - C_S[kb])
        dS[b] = alpha_h * e_T * s_short + beta_h * (S_T[b] - S_S[kb])
        dW[b] = alpha_h * e_T * v_T[b]  + beta_h * (W_T[b] - W_S[min(b, len(W_S)-1)])

    return dC, dS, dW   # Eq. (25)–(26)


def generate_parameter_groups(x_T, y_T, y_Td, v_T, C_T, S_T, W_T,
                               C_S, S_S, W_S, alphas, betas, lr_T):
    """
    Eqs. (24)–(27): Generate H sets of updated parameters, one per framework h.

    Parameters
    ----------
    alphas : (H,)  data-driven weights
    betas  : (H,)  knowledge-driven weights
    lr_T   : float learning rate for target FNN

    Returns
    -------
    groups : list of H dicts, each with keys 'C', 'S', 'W'
    """
    groups = []
    for alpha_h, beta_h in zip(alphas, betas):
        dC, dS, dW = compute_gradients(x_T, y_T, y_Td, v_T,
                                        C_T, S_T, W_T,
                                        C_S, S_S, W_S,
                                        alpha_h, beta_h)  # Eqs. 25–27
        # Eq. (24): Theta_T^h(t+1) = Theta_T(t) - lr * dE/dTheta_T
        C_h = C_T - lr_T * dC
        S_h = np.maximum(S_T - lr_T * dS, 0.01)
        W_h = W_T - lr_T * dW
        groups.append({'C': C_h, 'S': S_h, 'W': W_h})
    return groups


def target_fnn_output(x, C, S, W):
    """Forward pass for the target FNN (Eq. 16 — same structure as Eq. 7)."""
    diff = x[None, :] - C
    u = np.exp(-0.5 * ((diff / (S + 1e-8)) ** 2).sum(axis=1))
    v = u / (u.sum() + 1e-10)
    return float(W @ v)


def value_function(groups, x_future, y_future_d):
    """
    Eq. (28): Evaluate each framework h using future samples.

    Parameters
    ----------
    groups     : list of H dicts {'C', 'S', 'W'}
    x_future   : (N, Q) future input window
    y_future_d : (N,)   corresponding desired outputs

    Returns
    -------
    W_vals : (H,)  evaluation weights (mean squared future error per framework)
    """
    W_vals = np.zeros(len(groups))
    for h, g in enumerate(groups):
        errors = []
        for xf, yfd in zip(x_future, y_future_d):
            y_pred = target_fnn_output(xf, g['C'], g['S'], g['W'])   # Eq. (29)
            errors.append((y_pred - yfd) ** 2)
        W_vals[h] = np.mean(errors)   # Eq. (28): E[e_T^h(t),...,e_T^h(t+N)]
    return W_vals


# ---- Quick smoke-test ----
H = 5
alphas_h = np.linspace(0.5, 1.0, H)
betas_h  = 1.0 - alphas_h

# Re-run target forward pass to get v_T
y_T_fwd, u_T, v_T = source_fnn_forward(x_T, C_T, S_T, W_T)

groups = generate_parameter_groups(x_T, y_T_fwd, y_Td_val, v_T,
                                    C_T, S_T, W_T,
                                    C_s, S_s, W_s,
                                    alphas_h, betas_h, lr_T=0.01)  # Eqs. 24–27

# Simulate a small future window for Eq. (28)
x_future   = np.random.rand(3, Q)
y_future_d = np.random.rand(3)
W_vals = value_function(groups, x_future, y_future_d)             # Eq. 28

print(f"Eqs.(24–27) Generated {len(groups)} parameter groups.")
print(f"Eq.(28)     Value function W = {np.round(W_vals, 4)}")

---
## Step 13 — Eqs. (29)–(30): Select Optimal Parameters

**Eq. (29)** — Future prediction error for framework $h$ at horizon $n$:
$$e^h_T(t+n) = y_T\bigl(\Theta^h_T(t+1),\, x(t+n)\bigr) - y_{Td}(t+n)$$

**Eq. (30)** — Select the framework with minimum evaluation weight:
$$\Theta_T(t+1) = \Theta^h_T(t+1) \;\big|\; W^h(t) = \min W(t)$$

In [ ]:
# ============================================================
# Eqs. (29)–(30): Select optimal parameter group
# ============================================================

def select_optimal_parameters(groups, W_vals):
    """
    Eq. (30): Choose the framework h* that minimises the value function W^h(t).

    Parameters
    ----------
    groups : list of H dicts {'C', 'S', 'W'}
    W_vals : (H,)  evaluation weights from Eq. (28)

    Returns
    -------
    best   : dict  optimal {'C', 'S', 'W'}  — Theta_T(t+1)
    h_star : int   index of selected framework
    """
    h_star = int(np.argmin(W_vals))   # Eq. (30): arg min W(t)
    best   = groups[h_star]           # Theta_T(t+1)
    return best, h_star


# Eq. (29) is evaluated inside value_function() above as e_T^h(t+n)

# ---- Quick smoke-test ----
best_params, h_star = select_optimal_parameters(groups, W_vals)  # Eq. 30

# Apply optimal parameters back to the target FNN
C_T = best_params['C']
S_T = best_params['S']
W_T = best_params['W']

print(f"Eq.(29) Future errors evaluated for {len(groups)} frameworks.")
print(f"Eq.(30) Optimal framework selected: h* = {h_star}  "
      f"(W^h* = {W_vals[h_star]:.6f})")
print(f"        Target FNN updated — C_T shape: {C_T.shape}, W_T: {W_T.round(4)}")